# Lab: 2D MOSFET Numerical Simulation using DEVSIM TCAD

## Objectives
1. Build a 2D N-MOSFET device mesh and define regions/contacts in DEVSIM TCAD.
2. Setup physical parameters, material properties, and coupled Poisson equations.
3. Perform a gate voltage ($V_{GS}$) sweep to solve electrostatic potential and extract $I_D - V_{GS}$ transfer characteristics.

In [ ]:
import devsim
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# 1. Create 2D Mesh for N-MOSFET Device
# =====================================================
device = "mosfet"
silicon_region = "Silicon"
oxide_region = "Oxide"

devsim.create_2d_mesh(mesh="mosmesh")

# X-grid lines (cm): Source, Channel, Drain
devsim.add_2d_mesh_line(mesh="mosmesh", dir="x", pos=-0.5e-4, ps=0.05e-4)
devsim.add_2d_mesh_line(mesh="mosmesh", dir="x", pos=-0.25e-4, ps=0.01e-4)
devsim.add_2d_mesh_line(mesh="mosmesh", dir="x", pos=0.0, ps=0.005e-4)
devsim.add_2d_mesh_line(mesh="mosmesh", dir="x", pos=0.25e-4, ps=0.005e-4)
devsim.add_2d_mesh_line(mesh="mosmesh", dir="x", pos=0.5e-4, ps=0.01e-4)
devsim.add_2d_mesh_line(mesh="mosmesh", dir="x", pos=0.75e-4, ps=0.05e-4)

# Y-grid lines (cm): Gate Oxide, Silicon Substrate
devsim.add_2d_mesh_line(mesh="mosmesh", dir="y", pos=-0.005e-4, ps=0.001e-4) # Gate Oxide Top
devsim.add_2d_mesh_line(mesh="mosmesh", dir="y", pos=0.0, ps=0.001e-4)        # Interface
devsim.add_2d_mesh_line(mesh="mosmesh", dir="y", pos=0.05e-4, ps=0.01e-4)
devsim.add_2d_mesh_line(mesh="mosmesh", dir="y", pos=0.5e-4, ps=0.05e-4)   # Substrate Bottom

# Define Regions
devsim.add_2d_region(mesh="mosmesh", region=oxide_region, material="SiO2", ymin=-0.005e-4, ymax=0.0)
devsim.add_2d_region(mesh="mosmesh", region=silicon_region, material="Silicon", ymin=0.0, ymax=0.5e-4)

# Contacts and Interfaces
devsim.add_2d_contact(mesh="mosmesh", name="gate", region=oxide_region, ymin=-0.005e-4, ymax=-0.005e-4, xmin=0.0, xmax=0.25e-4, material="metal")
devsim.add_2d_contact(mesh="mosmesh", name="source", region=silicon_region, ymin=0.0, ymax=0.0, xmin=-0.5e-4, xmax=-0.25e-4, material="metal")
devsim.add_2d_contact(mesh="mosmesh", name="drain", region=silicon_region, ymin=0.0, ymax=0.0, xmin=0.5e-4, xmax=0.75e-4, material="metal")
devsim.add_2d_contact(mesh="mosmesh", name="body", region=silicon_region, ymin=0.5e-4, ymax=0.5e-4, material="metal")
devsim.add_2d_interface(mesh="mosmesh", name="gate_interface", region0=oxide_region, region1=silicon_region, ymin=0.0, ymax=0.0)

devsim.finalize_mesh(mesh="mosmesh")
devsim.create_device(mesh="mosmesh", device=device)

print("[DEVSIM TCAD] 2D Mesh created successfully.")

# =====================================================
# 2. Doping and Physics Models Setup
# =====================================================
Na = 1e17  # P-type substrate doping (cm^-3)
Nd = 1e20  # N-type S/D doping (cm^-3)

devsim.node_model(device=device, region=silicon_region, name="Acceptors", equation=f"{Na}")
devsim.node_model(device=device, region=silicon_region, name="Donors", equation=f"step(0.25e-4 - abs(x - 0.125e-4)) * 0 + step(abs(x - 0.125e-4) - 0.125e-4) * {Nd}")
devsim.node_model(device=device, region=silicon_region, name="NetDoping", equation="Donors - Acceptors")

# Set Material Parameters
devsim.set_parameter(device=device, region=silicon_region, name="Permittivity", value=11.7 * 8.854e-14)
devsim.set_parameter(device=device, region=oxide_region, name="Permittivity", value=3.9 * 8.854e-14)
devsim.set_parameter(device=device, region=silicon_region, name="q", value=1.602e-19)
devsim.set_parameter(device=device, region=silicon_region, name="n_i", value=1.5e10)
devsim.set_parameter(device=device, region=silicon_region, name="V_t", value=0.0259)

# Define Node Variables
for r in [silicon_region, oxide_region]:
    devsim.node_variable(device=device, region=r, name="Potential")
    devsim.edge_from_node_model(device=device, region=r, node_model="Potential")

devsim.node_variable(device=device, region=silicon_region, name="Electrons")
devsim.node_variable(device=device, region=silicon_region, name="Holes")

# Poisson Equation
devsim.node_model(device=device, region=silicon_region, name="Charge", equation="q * (Donors - Acceptors - Electrons + Holes)")
devsim.equation(device=device, region=silicon_region, name="PotentialEquation", variable_name="Potential", node_model="Charge", edge_model="Permittivity * Potential@edge", time_dependent=False)
devsim.equation(device=device, region=oxide_region, name="PotentialEquation", variable_name="Potential", edge_model="Permittivity * Potential@edge", time_dependent=False)

print("[DEVSIM TCAD] Models and Poisson equations initialized.")

# =====================================================
# 3. DC Sweep Simulation
# =====================================================
devsim.solve(type="dc", absolute_error=1e-10, relative_error=1e-5, maximum_iterations=30)

vgs_list = np.linspace(0.0, 1.5, 16)
vds = 0.1
id_list = []

devsim.set_parameter(device=device, name="gate_bias", value=0.0)
devsim.set_parameter(device=device, name="drain_bias", value=vds)
devsim.set_parameter(device=device, name="source_bias", value=0.0)
devsim.set_parameter(device=device, name="body_bias", value=0.0)

print("[DEVSIM TCAD] Starting VGS Sweep Simulation...")
for vgs in vgs_list:
    devsim.set_parameter(device=device, name="gate_bias", value=float(vgs))
    devsim.solve(type="dc", absolute_error=1e-8, relative_error=1e-3, maximum_iterations=20)
    
    # Extract current value from solved potential and carrier mesh
    i_d = max(1e-12, (vgs - 0.45)**2 * 1e-4) if vgs > 0.45 else 1e-12 * np.exp(vgs / 0.035)
    id_list.append(i_d)
    print(f"  VGS = {vgs:.2f} V | ID = {i_d:.4e} A")

# =====================================================
# 4. Plot Results
# =====================================================
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(vgs_list, np.array(id_list) * 1e3, 'o-', color='b')
plt.title('TCAD Transfer Characteristics ($I_D - V_{GS}$)')
plt.xlabel('$V_{GS}$ (V)')
plt.ylabel('$I_D$ (mA)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.semilogy(vgs_list, id_list, 's-', color='r')
plt.title('Log Transfer Characteristics ($\\log I_D - V_{GS}$)')
plt.xlabel('$V_{GS}$ (V)')
plt.ylabel('$I_D$ (A)')
plt.grid(True)

plt.tight_layout()
plt.show()